In [32]:
import pandas as pd 

In [46]:
strat_train_set = pd.read_pickle("../dataset/strat_train_set")
strat_val_set = pd.read_pickle("../dataset/strat_val_set")

strat_train_set['Irrigation_Need'] = strat_train_set['class_enc']
strat_val_set['Irrigation_Need'] = strat_val_set['class_enc']
strat_train_set.drop("class_enc", axis=1, inplace=True)
strat_val_set.drop("class_enc", axis=1, inplace=True)

strat_train_set["Mulching_Used"] = (strat_train_set["Mulching_Used"] == 1).astype(int)
strat_val_set["Mulching_Used"] = (strat_val_set["Mulching_Used"] == 1).astype(int)

In [ ]:
# from sklearn.base import BaseEstimator, TransformerMixin
# from sklearn.preprocessing import OneHotEncoder

# class FeatureEncoder (BaseEstimator, TransformerMixin):
#     def fit (self, X, y = None): 
#         return self 
    
#     def transform (self, X : pd.DataFrame, y = None ):
#         X["Mulching_Used"] = (X["Mulching_Used"] == 1 ).astype(int)
#         X["Irrigation_Need"] = X["class_enc"]
#         encoder = OneHotEncoder ()
#         for col in X.select_dtypes(include="object").columns:
#             matrix = encoder.fit_transform(X[[col]]).toarray()

#             column_names = X[col].unique()
#             for i in range(len(matrix.T)):
#                 X[column_names[i]] = matrix.T[i]


#         return X


            

        

        


In [48]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder
class FeatureEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.encoders_ = {}
        self.column_names_ = {}
        for col in X.select_dtypes(include="object").columns:
            enc = OneHotEncoder(sparse_output=False)
            enc.fit(X[[col]])
            self.encoders_[col] = enc
            self.column_names_[col] = enc.categories_[0]  # consistent order
        return self

    def transform(self, X: pd.DataFrame, y=None):
        X = X.copy()
        # X["Irrigation_Need"] = X["class_enc"]
        for col, enc in self.encoders_.items():
            matrix = enc.transform(X[[col]])
            for i, name in enumerate(self.column_names_[col]):
                X[name] = matrix[:, i]
        return X

In [ ]:
class FeatureDropper (BaseEstimator, TransformerMixin):
    def fit (self, X, y=None):
        return self 
    
    def transform (self, X : pd.DataFrame): 
        dropped_cols = []
        for col in X.select_dtypes(include="object").columns:
            dropped_cols.append(col)

        return X.drop(dropped_cols, axis = 1, errors = "ignore")

    

In [50]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([("feature_encoder", FeatureEncoder()), ("feature_dropper", FeatureDropper())])

In [45]:
strat_train_set.info()

<class 'pandas.core.frame.DataFrame'>
Index: 504000 entries, 434935 to 619674
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       504000 non-null  int64  
 1   Soil_Type                504000 non-null  object 
 2   Soil_pH                  504000 non-null  float64
 3   Soil_Moisture            504000 non-null  float64
 4   Organic_Carbon           504000 non-null  float64
 5   Electrical_Conductivity  504000 non-null  float64
 6   Temperature_C            504000 non-null  float64
 7   Humidity                 504000 non-null  float64
 8   Rainfall_mm              504000 non-null  float64
 9   Sunlight_Hours           504000 non-null  float64
 10  Wind_Speed_kmh           504000 non-null  float64
 11  Crop_Type                504000 non-null  object 
 12  Crop_Growth_Stage        504000 non-null  object 
 13  Season                   504000 non-null  object 
 14  Irri

In [51]:
strat_train_set = pipeline.fit_transform(strat_train_set)

In [52]:
strat_train_set

,id,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,...,Sprinkler,Groundwater,Rainwater,Reservoir,River,Central,East,North,South,West
434935,434935,7.84,48.52,0.58,1.96,39.36,89.96,1340.68,10.72,14.58,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
581347,581347,5.84,31.38,0.57,0.78,13.54,45.75,2223.23,10.26,17.69,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
274983,274983,5.14,43.06,1.05,0.53,31.83,75.56,759.03,4.20,5.88,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
193010,193010,5.57,33.60,0.66,1.63,40.92,34.63,2094.61,9.65,2.21,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
433470,433470,6.44,63.65,1.00,2.59,39.94,54.24,990.91,6.56,18.10,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
324010,324010,5.73,54.06,0.34,1.46,28.69,79.94,119.68,4.53,5.49,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
540194,540194,5.72,13.91,0.72,0.24,23.94,76.42,2173.52,9.76,18.71,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
517771,517771,6.82,35.90,0.98,0.41,36.55,91.73,2153.19,5.37,8.42,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
214439,214439,5.97,32.97,1.26,2.34,21.98,52.35,1707.39,9.32,10.22,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0


In [53]:
from sklearn.preprocessing import StandardScaler

X = strat_train_set.drop("Irrigation_Need", axis=1)
Y = strat_train_set["Irrigation_Need"]

scaler = StandardScaler()
X_data = scaler.fit_transform(X)
Y_data = Y.to_numpy()

In [ ]:
X_data

array([[ 0.65926278,  1.47183804,  0.68440716, ...,  2.129253  ,
        -0.52243605, -0.51248   ],
       [ 1.46462946, -0.69580694, -0.36190975, ..., -0.46964828,
        -0.52243605, -0.51248   ],
       [-0.22058321, -1.45448269,  0.35109967, ...,  2.129253  ,
        -0.52243605, -0.51248   ],
       ...,
       [ 1.11491774,  0.3663391 , -0.08598487, ...,  2.129253  ,
        -0.52243605, -0.51248   ],
       [-0.55361684, -0.55491002, -0.26484768, ..., -0.46964828,
        -0.52243605, -0.51248   ],
       [ 1.67545431, -0.1105428 ,  1.59398253, ...,  2.129253  ,
        -0.52243605, -0.51248   ]], shape=(504000, 43))

In [56]:
Y_data

array([1, 0, 0, ..., 0, 0, 1], shape=(504000,))

In [55]:

# 2. Preprocess the validation set (using the SAME pipeline, don't refit)
strat_val_set = pipeline.transform(strat_val_set)

X_val = strat_val_set.drop("Irrigation_Need", axis=1)
Y_val = strat_val_set["Irrigation_Need"]

X_val_scaled = scaler.transform(X_val)  # use the already-fitted scaler

/Users/mohamed/irrigation/venv/lib/python3.10/site-packages/sklearn/pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


Done with preprocessing
Now we have: 
- X_data 
- Y_data 
- X_val_scaled 
- Y_val

## Random Forest 

First train with some inital hyperparameters

In [64]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# use random_state=42 for reproducibility
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_data, Y_data)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


Evaluate using balanced accuracy score which is used for unbalanced classes

In [ ]:
from sklearn.metrics import balanced_accuracy_score

Y_pred = model.predict(X_val_scaled)
score = balanced_accuracy_score(Y_val, Y_pred)
print(f"Balanced Accuracy: {score:.4f}")

Balanced Accuracy: 0.8381


## XGBoost

In [70]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import balanced_accuracy_score, make_scorer

# XGBoost needs numeric labels starting from 0
# If Y_data contains 0 and 1 already, you're fine
# If not, encode it:
# from sklearn.preprocessing import LabelEncoder
# Y_data = LabelEncoder().fit_transform(Y_data)

param_dist = {
    "n_estimators": [100, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [5, 10, 20],
    "subsample": [0.7, 0.8, 1.0],          # fraction of rows per tree
    "colsample_bytree": [0.7, 0.8, 1.0],   # fraction of features per tree
    "scale_pos_weight": [1, 5, 10]          # handles class imbalance
}

scorer = make_scorer(balanced_accuracy_score)

search = RandomizedSearchCV(
    XGBClassifier(
        tree_method="hist",     # faster tree building
        device="cpu",           # change to "cuda" if you have a GPU
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    ),
    param_distributions=param_dist,
    n_iter=20,
    scoring=scorer,
    cv=3,                       # 3 folds to keep it fast
    verbose=2,
    random_state=42
)

search.fit(X_data, Y_data)

print("Best params:", search.best_params_)
print("Best CV balanced accuracy:", search.best_score_)

Y_pred = search.best_estimator_.predict(X_val_scaled)
print("Val balanced accuracy:", balanced_accuracy_score(Y_val, Y_pred))

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/mohamed/irrigation/venv/lib/python3.10/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <1A0D8152-BF46-3BE0-B651-EE965C187777> /Users/mohamed/irrigation/venv/lib/python3.10/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dyl